# 04 — LLM Knowledge Extraction
Uses the local LLM to mine training pairs from real Examples, Book chapters, and Proposals.
All generated ARO code is validated with `aro check`. Valid pairs are saved to
`knowledge_pairs.jsonl`.

Run **`05_warmstart_finetune.ipynb`** after this to warm-start fine-tune the model
on these pairs before the execution-grounded generation in
`07_repl_execution_training.ipynb`.

**Input:**  `../data/02_knowledge/knowledge.json`, `../../Examples/`, `../../Book/`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl`

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, random, subprocess, tempfile
from pathlib import Path
from collections import Counter

DATA_OUT    = DATA_ROOT / '02_knowledge'

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

print(f'ARO root: {ARO_ROOT}')
print(f'Knowledge base: {len(kb["actions"])} actions, {len(kb["examples"])} examples, {len(kb["proposals"])} proposals')
print(f'Loading {MODEL_ID}...')
model, tokenizer, _, mlx_generate, make_sampler = load_model()
print('Model ready.')

In [ ]:
# Build system prompt from action metadata
action_lines = []
for a in kb['actions']:
    if a['verbs']:
        v = '/'.join(a['verbs'][:3])
        p = ', '.join(a['prepositions'][:4])
        action_lines.append(f'- {v}  (role: {a["role"]}, prepositions: {p})')

SYSTEM_PROMPT = """You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }

KEY RULES:
- Articles (a/an/the) are optional everywhere
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Forbidden variable name prefixes: is-, with-, empty-
- Variables are IMMUTABLE — bind once; use a new name for each transformation
- Exactly ONE Application-Start per application (error if 0 or multiple)
- openapi.yaml REQUIRED for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.

HTTP:
- Path params:   Extract the <id> from the <pathParameters: id>.
- Request body:  Extract the <data> from the <request: body>.
- Return statuses: <OK: status>, <Created: status>, <NoContent: status>,
                   <NotFound: status>, <BadRequest: status>, <Conflict: status>,
                   <Unauthorized: status>, <InternalServerError: status>

CONTROL FLOW:
- Conditional:   when <condition> { statements }
- Loop:          For each <item> in <list> { statements }
- Match:         match <var> { case value { statements } case other { statements } }
- Guard on declaration: (Name: EventName Handler) when <field> = value { ... }

COMPUTE & ARITHMETIC:
- Compute the <total> from <price> * <qty>.
- Compute the <upper: uppercase> from <text>.
- Compute the <len: length> from <text>.
- Supported ops: +, -, *, /, % (integers); ++ (strings)

CROSS-FEATURE SHARING:
- Publish as <alias> <variable>.  (makes variable accessible across feature sets in same business activity)

EVENTS:
- Emit a <EventName: event> with <data>.
- Handler:  (HandlerName: EventName Handler) { ... }
- State:    Accept the <new-state: toState> for the <entity>.

LIFECYCLE:
- Application-Start: required entry point
- Application-End: Success (optional graceful shutdown)
- Application-End: Error (optional crash handler)

AVAILABLE ACTIONS (verb → role → prepositions):
- Extract     REQUEST   from/with     — pull data from request/event/object
- Retrieve    REQUEST   from/where    — fetch from repository or service
- Fetch       REQUEST   from/with     — HTTP GET external URL
- Parse       REQUEST   from          — parse JSON/YAML/HTML/CSV text
- Request     REQUEST   from/with     — HTTP POST/PUT/PATCH
- Compute     OWN       from/with/for — arithmetic, string ops, built-in transforms
- Transform   OWN       from/with     — type coercion (e.g. string → int)
- Validate    OWN       for/with      — check constraints, return bool
- Compare     OWN       against/with  — compare two values, return bool
- Create      OWN       with          — instantiate struct or entity
- Merge       OWN       with          — combine objects or collections
- Filter      OWN       where/with    — select subset of collection
- Sort        OWN       by/with       — order a collection
- Split       OWN       by/from       — split string or list by delimiter
- Join        OWN       with/from     — join strings or list elements
- Accept      OWN       with/for      — state machine transition
- Return      RESPONSE  with/for      — send HTTP response and end feature set
- Throw       RESPONSE  with/for      — raise exception (propagates as error)
- Store       EXPORT    into          — persist to repository (auto-generates id)
- Update      EXPORT    into          — update existing record in repository
- Delete      EXPORT    from          — remove record from repository
- Log         EXPORT    to            — write to console or log service
- Send        EXPORT    to/with       — send email or message
- Emit        EXPORT    with          — publish domain event to EventBus
- Publish     EXPORT    as            — share variable across feature sets
- Start       EXPORT    with          — start a service (HTTP server, file monitor)
- Stop        EXPORT    with          — stop a running service
- Keepalive   EXPORT    for           — block until shutdown signal (for servers)
- Render      EXPORT    with/using    — render Mustache template
- Notify      EXPORT    with          — send notification to user(s)
- Configure   EXPORT    with          — set runtime configuration option
- Read        REQUEST   from          — read file contents
- Write       EXPORT    to            — write data to file
- Copy        EXPORT    to            — copy file to destination
- Move        EXPORT    to            — move file to destination

Always wrap ARO code in ```aro ... ``` fences.
Always generate complete, valid ARO that would pass `aro check`.
"""


def chat(messages, max_tokens=800):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=0.3),
    )


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def count_aro_statements(code):
    """Count meaningful ARO statements in a code block (excludes comments, braces, headers)."""
    lines = [l.strip() for l in code.strip().split('\n')]
    return sum(1 for l in lines
               if l and not l.startswith('(*') and not l.startswith('*)')
               and not l.startswith('}') and not l.startswith('(')
               and not l == '{' and l.endswith('.'))


def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:500]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def aro_run_quick(code, timeout=8):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'run', tmp], capture_output=True, text=True, timeout=timeout)
            return r.returncode == 0, (r.stdout + r.stderr).strip()[:200]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return True, 'server_timeout'


def auto_wrap_aro(code):
    """Wrap bare ARO statements in a feature set if they don't already have one.

    Many proposal code blocks are bare snippets like:
        Extract the <user> from the <request: body>.
        Compute the <total> from <price> * <qty>.
    These need a (Name: Activity) { ... } wrapper to pass aro check.

    Returns (wrapped_code, was_wrapped) or (None, False) if the block is
    template/meta-syntax/negative-example that should be skipped.
    """
    stripped = code.strip()

    # Already has a feature set wrapper — use as-is
    if stripped.startswith('(') and '{' in stripped.split('\n')[0]:
        return code, False

    # Filter out non-code content that can't be wrapped
    # Template placeholders
    if '<statements>' in stripped or '<statement' in stripped.lower():
        return None, False
    # Annotation/diagram markers
    if '^^^^' in stripped or '───' in stripped:
        return None, False
    # Negative examples showing deliberate errors
    if '❌' in stripped:
        return None, False
    # Meta-syntax descriptions (e.g. "Action the <result> preposition the <object>")
    if re.match(r'^[A-Z][a-z]+ the <[a-z]+> [a-z]+ the <[a-z]+>\s*$', stripped.split('\n')[0].strip().rstrip('.')):
        if 'preposition' in stripped or 'qualifier' in stripped:
            return None, False
    # C-style comments instead of ARO comments
    if '//' in stripped and '(*' not in stripped:
        return None, False

    # Check if any line looks like a real ARO statement (has angle-bracket variables)
    has_aro_stmt = any(re.search(r'<[a-z][-a-z0-9]*', line)
                       for line in stripped.split('\n')
                       if not line.strip().startswith('(*'))
    if not has_aro_stmt:
        return None, False

    # Filter out blocks that already have a Return/Throw (don't double-add)
    has_return = any(re.match(r'\s*(Return|Throw)\b', line)
                     for line in stripped.split('\n'))

    # Indent the bare statements
    indented = '\n'.join(f'    {line}' for line in stripped.split('\n'))

    if has_return:
        wrapped = f'(Application-Start: Example) {{\n{indented}\n}}'
    else:
        wrapped = f'(Application-Start: Example) {{\n{indented}\n    Return an <OK: status> for the <result>.\n}}'

    return wrapped, True


def repair_aro(code, error, context_messages, max_retries=2):
    """Try to fix ARO code that failed aro check by feeding the error back to the LLM.

    Returns (fixed_code, check_ok, attempts) or (None, False, attempts) if all retries fail.
    """
    current_code = code
    for attempt in range(max_retries):
        repair_messages = context_messages + [
            {'role': 'assistant', 'content': f'```aro\n{current_code}\n```'},
            {'role': 'user', 'content': (
                f'This ARO code failed validation with error:\n{error}\n\n'
                f'Fix the code so it passes `aro check`. Reply with ONLY the corrected code '
                f'in ```aro ... ``` fences. Do not explain.'
            )},
        ]
        fixed_text = chat(repair_messages, max_tokens=800)
        fixed_blocks = extract_aro_blocks(fixed_text)
        if not fixed_blocks:
            return None, False, attempt + 1

        current_code = fixed_blocks[0]
        check_ok, check_err = aro_check(current_code)
        if check_ok is True:
            return current_code, True, attempt + 1
        if check_ok is None:
            return current_code, None, attempt + 1
        error = check_err  # update error for next retry

    return None, False, max_retries


def clean_instruction(instruction):
    """Post-process instruction to remove ARO jargon and improve diversity."""
    # Remove angle-bracket ARO syntax from instructions (these are code, not instructions)
    if re.search(r'<[a-z][-a-z0-9]*(?::\s*[a-z][-a-z0-9]*)?\s*>', instruction):
        return None  # reject instructions that ARE ARO syntax

    # Replace ARO jargon with natural phrasing
    replacements = [
        (r'\bfeature set\b', 'code'),
        (r'\bbusiness activity\b', 'module'),
    ]
    cleaned = instruction
    for pattern, repl in replacements:
        cleaned = re.sub(pattern, repl, cleaned, flags=re.IGNORECASE)

    return cleaned.strip()


# ── Pipeline statistics tracking ─────────────────────────────────────────────
pipeline_stats = {
    phase: {'attempted': 0, 'valid': 0, 'repaired': 0, 'wrapped': 0,
            'dropped_check': 0, 'dropped_instr': 0, 'dropped_trivial': 0,
            'dropped_truncated': 0, 'dropped_misaligned': 0}
    for phase in ('Phase 1', 'Phase 2', 'Phase 3', 'Phase 4', 'Variants')
}


# Resume support
_done_keys = set()
_pair_count = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    s = json.loads(line)
                    if s.get('notebook') == 'NB04':
                        _done_keys.add((s['source'], s['instruction'][:80]))
                        _pair_count += 1
                except Exception:
                    pass
    if _pair_count:
        print(f'Resuming — {_pair_count} NB04 pairs already saved')


_judge_drops = {'count': 0}

def _nb03_judge_chat(msgs, max_tokens=120, temp=0.0):
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return mlx_generate(
        model, tokenizer,
        prompt=tokenizer.encode(text),
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=temp),
    ).strip()

def save_pair(source, instruction, output, score=1.0, metadata=None,
              skip_gate=False):
    key = (source, instruction[:80])
    if key in _done_keys:
        return False
    # Semantic alignment gate (audit fix): drop pairs whose code
    # doesn't address the instruction. Same judge as NB10. Only
    # runs on pairs that have an ```aro``` block — prose Q&A
    # passes through. Callers that already judged the pair themselves
    # (Phase 5 variants, issue #381) pass skip_gate=True.
    if '```aro' in (output or '') and not skip_gate:
        aligned, _reason = semantic_alignment_check(
            instruction, output, _nb03_judge_chat,
        )
        if not aligned:
            _judge_drops['count'] += 1
            if _judge_drops['count'] <= 5:
                print(f'  NB04 semantic-gate drop: {_reason[:120]}', flush=True)
            return False
    record = {
        'instruction': instruction,
        'output': output,
        'source': source,
        'score': score,
        'metadata': metadata or {},
        'notebook': 'NB04',
    }
    # save_notebook_pair applies the shared FIXTRAIN lint gate (issue #410)
    # and returns False when the pair was dropped.
    if not save_notebook_pair('NB04', record):
        _done_keys.add(key)
        return False
    _done_keys.add(key)
    return True


print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print('Helpers ready.')

## Phase 1 — Real Examples → Training Pairs

For each real, validated ARO example:
- Try up to 3 different prompt styles to generate a natural language instruction
- Falls back to README.md content if LLM instruction generation fails
- Rejects instructions containing ARO syntax (angle brackets) or jargon
- The code itself is already valid (taken verbatim from the repository)
- Result: (instruction, real_code) pairs with score=1.0

This is the highest-quality training data because the code is guaranteed correct.

In [ ]:
clean_notebook_pairs('NB04')

def load_app_dir(app_path):
    """Load a single ARO application directory into the same dict shape as kb['examples']."""
    p = Path(app_path)
    aro_files = {}
    for f in sorted(p.rglob('*.aro')):
        rel = str(f.relative_to(p))
        aro_files[rel] = f.read_text()

    def read_opt(name):
        fp = p / name
        return fp.read_text() if fp.exists() else None

    return {
        'name':      p.name,
        'dir':       str(p),
        'aro_files': aro_files,
        'openapi':   read_opt('openapi.yaml'),
        'readme':    read_opt('README.md'),
        'expected':  None,
    }


# ── Collect all examples ─────────────────────────────────────────────────────
all_examples = list(kb['examples'])

# ARO-Application root comes from config (override: ARO_APPLICATION_PATH env var).
ARO_APP_DIR = ARO_APPLICATION_ROOT
if ARO_APP_DIR.exists():
    known_names = {e['name'] for e in all_examples}
    for app_subdir in sorted(ARO_APP_DIR.iterdir()):
        if not app_subdir.is_dir():
            continue
        if app_subdir.name in known_names:
            continue
        if not list(app_subdir.rglob('*.aro')):
            continue
        all_examples.append(load_app_dir(app_subdir))
        print(f'  Added ARO-Application/{app_subdir.name}', flush=True)
else:
    print(f'  ARO-Application not found at {ARO_APP_DIR}')

print(f'\nTotal examples for Phase 1: {len(all_examples)} '
      f'({len(kb["examples"])} from knowledge base + '
      f'{len(all_examples) - len(kb["examples"])} from ARO-Application)')


# ── Instruction prompt variants for diversity ────────────────────────────────
_INSTR_PROMPTS = [
    (
        'Write ONE clear, specific natural language instruction (2-3 sentences) that '
        'describes exactly what this application does — specific enough that a developer '
        'following the instruction would produce this code.\n\n'
        'Reply with ONLY the instruction text. No markdown, no labels.'
    ),
    (
        'Imagine a developer asks you to build this application. '
        'Write the request they would make (2-3 sentences). '
        'Use plain English, no ARO-specific terms like "feature set" or angle brackets.\n\n'
        'Reply with ONLY the request text.'
    ),
    (
        'Describe what this application does as a task description for a programmer. '
        'Be specific about the data flow and operations. '
        'Do NOT use ARO jargon like "feature set", "business activity", or angle-bracket syntax.\n\n'
        'Reply with ONLY the description (2-3 sentences).'
    ),
]


# ── Generate instruction for each example ────────────────────────────────────
print(f'\n--- Phase 1: Real Examples → Training Pairs ---')
count = 0
truncated_flagged = 0
stats = pipeline_stats['Phase 1']

for ex in all_examples:
    if not ex['aro_files']:
        continue

    source_key = f'example:{ex["name"]}'
    stats['attempted'] += 1

    # Build a compact code representation for the LLM context window.
    # Multi-file apps get a file manifest FIRST so the model knows what it
    # is (not) seeing, and truncation is measured and recorded (issue #379) —
    # silently-truncated apps teach the model that complex applications fit
    # in 4 KB.
    num_aro_files = len(ex['aro_files'])
    total_code_chars = sum(len(c) for c in ex['aro_files'].values())
    manifest_lines = [f'This application contains {num_aro_files} .aro file(s):']
    for fname, code in ex['aro_files'].items():
        manifest_lines.append(f'  - {fname} ({len(code):,} chars)')
    if ex.get('openapi'):
        manifest_lines.append(f'  - openapi.yaml ({len(ex["openapi"]):,} chars)')

    parts = []
    if num_aro_files > 1:
        parts.append('\n'.join(manifest_lines))
    if ex.get('openapi'):
        parts.append(f'## openapi.yaml\n```yaml\n{ex["openapi"][:600]}\n```')
    shown_files = list(ex['aro_files'].items())[:5]
    for fname, code in shown_files:
        parts.append(f'## {fname}\n```aro\n{code[:1200]}\n```')
    if num_aro_files > len(shown_files):
        parts.append(f'(… {num_aro_files - len(shown_files)} more .aro files '
                     f'not shown — see the file manifest above)')
    full_repr = '\n\n'.join(parts)[:4000]

    shown_code_chars = sum(min(len(c), 1200) for _, c in shown_files)
    truncated = (num_aro_files > len(shown_files)) or (shown_code_chars < total_code_chars)
    coverage = shown_code_chars / max(1, total_code_chars)

    # Severely truncated representations produce instructions that describe
    # a fraction of the app — exclude them from training entirely.
    if truncated and coverage < 0.4:
        stats['dropped_truncated'] += 1
        print(f'    skip {ex["name"]}: severely truncated multi-file app '
              f'({num_aro_files} files, only {coverage:.0%} visible to the LLM)',
              flush=True)
        continue
    if truncated:
        truncated_flagged += 1

    # Reconstruct full output (all .aro files + openapi)
    output_parts = []
    if ex.get('openapi'):
        output_parts.append(f'## openapi.yaml\n```yaml\n{ex["openapi"]}\n```')
    for fname, code in ex['aro_files'].items():
        output_parts.append(f'## {fname}\n```aro\n{code}\n```')
    output = '\n\n'.join(output_parts)

    # Try up to 3 prompt variants to get a good instruction
    instruction = None
    for prompt_idx, instr_prompt in enumerate(_INSTR_PROMPTS):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': (
                f'Here is a complete ARO application called "{ex["name"]}":\n\n{full_repr}\n\n'
                f'{instr_prompt}'
            )},
        ]
        raw_instruction = chat(messages, max_tokens=250).strip()
        raw_instruction = re.sub(r'^(#+\s*|Instruction:\s*|\*\*[^*]+\*\*:\s*)', '', raw_instruction, flags=re.MULTILINE).strip()
        raw_instruction = raw_instruction.split('\n')[0].strip()

        cleaned = clean_instruction(raw_instruction)
        if cleaned and len(cleaned) >= 20:
            instruction = cleaned
            break

    # Fallback: use README.md content as instruction basis
    if not instruction and ex.get('readme'):
        readme_text = ex['readme']
        # Extract first meaningful paragraph from README
        readme_lines = [l.strip() for l in readme_text.split('\n')
                        if l.strip() and not l.strip().startswith('#')
                        and not l.strip().startswith('```')
                        and len(l.strip()) > 20]
        if readme_lines:
            instruction = readme_lines[0][:300]

    if not instruction or len(instruction) < 20:
        stats['dropped_instr'] += 1
        print(f'    skip {ex["name"]}: no usable instruction after 3 attempts', flush=True)
        continue

    if save_pair(source_key, instruction, output, score=1.0,
                 metadata={'example': ex['name'], 'has_openapi': bool(ex.get('openapi')),
                            'source_dir': ex.get('dir', ''),
                            'num_aro_files': num_aro_files,
                            'truncated_representation': truncated,
                            'llm_context_coverage': round(coverage, 2)}):
        stats['valid'] += 1
        count += 1
        tag = ' [ARO-App]' if ARO_APP_DIR.exists() and ex['name'] in {e.name for e in ARO_APP_DIR.iterdir() if e.is_dir()} else ''
        print(f'  [{count}]{tag} {ex["name"]}: {instruction[:85]}', flush=True)

print(f'\nPhase 1 done: {count} new pairs from real examples')
print(f'  attempted={stats["attempted"]}, valid={stats["valid"]}, '
      f'dropped_instr={stats["dropped_instr"]}, '
      f'dropped_truncated={stats["dropped_truncated"]}')
print(f'  {truncated_flagged} kept pairs flagged truncated_representation '
      f'(manifest shown to LLM, full code in output)')

## Phase 2 — Book: TheLanguageGuide → Training Pairs

For each Language Guide chapter with ARO code blocks:
- Ask the LLM to generate 5 instruction→code pairs demonstrating chapter concepts
- Instructions must use plain English (no ARO jargon like "feature set")
- Code must have at least 2 ARO statements (rejects trivial single-statement wrappers)
- Validate each generated code block with `aro check`
- If validation fails, feed the error back to the LLM for repair (up to 2 attempts)
- Save only validated, non-trivial pairs

In [ ]:
print(f'\n--- Phase 2: TheLanguageGuide → Training Pairs ---')
count = 0
repaired = 0
wrapped = 0
stats = pipeline_stats['Phase 2']

guide_dir = ARO_ROOT / 'Book' / 'TheLanguageGuide'
chapters = sorted(guide_dir.glob('*.md'))

for chapter_path in chapters:
    chapter_name = chapter_path.stem
    content = chapter_path.read_text()

    code_blocks = extract_aro_blocks(content)
    if not code_blocks:
        print(f'  {chapter_name}: no ARO blocks, skipping', flush=True)
        continue

    content_trimmed = content[:4500]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is "{chapter_name}" from the ARO Language Guide:\n\n{content_trimmed}\n\n'
            f'Generate 5 distinct instruction → ARO code training pairs based on this chapter.\n'
            f'Each pair must demonstrate a DIFFERENT concept from the chapter.\n'
            f'The ARO code must be COMPLETE — wrapped in a feature set like:\n'
            f'  (Application-Start: Example) {{\n'
            f'      <statements>\n'
            f'      Return an <OK: status> for the <result>.\n'
            f'  }}\n'
            f'The code must have at least 3 statements and pass `aro check`.\n'
            f'Do NOT use ARO jargon like "feature set" or "business activity" in instructions.\n'
            f'Do NOT use angle-bracket syntax in instructions — use plain English.\n'
            f'Vary instruction style: use "Build...", "I need...", "Set up...", "How would I...", "Create...", etc.\n\n'
            f'Format EXACTLY like this:\n\n'
            f'### Pair 1\n'
            f'**Instruction:** <natural language instruction>\n'
            f'**Code:**\n'
            f'```aro\n<complete valid ARO code with feature set wrapper and 3+ statements>\n```\n\n'
            f'### Pair 2\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```\n\n'
            f'(continue through Pair 5)'
        )},
    ]

    output_text = chat(messages, max_tokens=2000)

    pair_blocks = re.split(r'###\s*Pair\s*\d+', output_text)
    chapter_count = 0

    for block in pair_blocks[1:]:
        stats['attempted'] += 1

        instr_match = re.search(r'\*\*Instruction:\*\*\s*(.+?)(?=\n\*\*|\n```|\Z)', block, re.DOTALL)
        if not instr_match:
            instr_match = re.search(r'Instruction:\s*(.+?)(?=\n(?:Code:|```|\*\*)|\Z)', block, re.DOTALL)
        if not instr_match:
            stats['dropped_instr'] += 1
            continue
        raw_instruction = instr_match.group(1).strip().replace('\n', ' ')

        codes = extract_aro_blocks(block)
        if not codes:
            stats['dropped_instr'] += 1
            continue

        instruction = clean_instruction(raw_instruction)
        if not instruction or len(instruction) < 15:
            stats['dropped_instr'] += 1
            continue

        code = codes[0]

        # Reject trivially short outputs (single-statement wrappers teach nothing)
        if count_aro_statements(code) < 2:
            stats['dropped_trivial'] += 1
            print(f'    ~ {chapter_name}: trivial ({count_aro_statements(code)} stmt)', flush=True)
            continue

        check_ok, check_err = aro_check(code)

        # Auto-wrap bare snippets (LLM often mimics the chapter's unwrapped style)
        if check_ok is False:
            wrap_result, was_wrapped = auto_wrap_aro(code)
            if wrap_result is not None and was_wrapped:
                wrap_ok, wrap_err = aro_check(wrap_result)
                if wrap_ok is True:
                    code = wrap_result
                    check_ok = True
                    wrapped += 1
                    stats['wrapped'] += 1

        # Repair loop
        if check_ok is False:
            fixed_code, check_ok, attempts = repair_aro(
                code, check_err, messages, max_retries=2
            )
            if check_ok is True:
                code = fixed_code
                repaired += 1
                stats['repaired'] += 1
                print(f'    ✓ {chapter_name}: repaired in {attempts} attempt(s)', flush=True)
            else:
                stats['dropped_check'] += 1
                short_err = check_err.split('\n')[0][:70]
                print(f'    x {chapter_name}: {short_err}', flush=True)
                continue

        score = 1.0 if check_ok is True else 0.8
        if save_pair(f'book:{chapter_name}', instruction, f'```aro\n{code}\n```',
                     score=score, metadata={'chapter': chapter_name}):
            stats['valid'] += 1
            chapter_count += 1
            count += 1
            print(f'  [{count}] {chapter_name}: {instruction[:80]}', flush=True)

    if chapter_count == 0:
        print(f'  {chapter_name}: 0 valid pairs extracted', flush=True)

print(f'\nPhase 2 done: {count} new pairs from Language Guide ({wrapped} wrapped, {repaired} repaired)')
print(f'  attempted={stats["attempted"]}, valid={stats["valid"]}, wrapped={stats["wrapped"]}, '
      f'repaired={stats["repaired"]}, dropped_check={stats["dropped_check"]}, '
      f'dropped_instr={stats["dropped_instr"]}, dropped_trivial={stats["dropped_trivial"]}')

## Phase 3 — Book: AROByExample → Training Pairs

AROByExample builds a complete web crawler in ARO across 14 chapters.
Each chapter shows real, working ARO code for a specific concern (fetching, parsing, storing, etc.).
The LLM generates 2 training pairs per chapter from the complete worked examples.

In [ ]:
print(f'\n--- Phase 3: AROByExample → Training Pairs ---')
count = 0
repaired = 0
wrapped = 0
stats = pipeline_stats['Phase 3']

aro_by_example_dir = ARO_ROOT / 'Book' / 'AROByExample'
chapters = sorted(aro_by_example_dir.glob('Chapter*.md'))

for chapter_path in chapters:
    chapter_name = chapter_path.stem
    content = chapter_path.read_text()
    code_blocks = extract_aro_blocks(content)
    if not code_blocks:
        continue

    content_trimmed = content[:4000]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is "{chapter_name}" from "ARO By Example" (a hands-on ARO tutorial):\n\n'
            f'{content_trimmed}\n\n'
            f'Generate 2 instruction → ARO code pairs from the examples in this chapter.\n'
            f'The code must be COMPLETE — wrapped in a feature set with 3+ statements.\n'
            f'Use plain English instructions — no ARO jargon or angle-bracket syntax.\n\n'
            f'### Pair 1\n'
            f'**Instruction:** <what this code does, in plain English>\n'
            f'**Code:**\n'
            f'```aro\n<the ARO code>\n```\n\n'
            f'### Pair 2\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```'
        )},
    ]

    output_text = chat(messages, max_tokens=1000)
    pair_blocks = re.split(r'###\s*Pair\s*\d+', output_text)

    for block in pair_blocks[1:]:
        stats['attempted'] += 1

        instr_match = re.search(r'\*\*Instruction:\*\*\s*(.+?)(?=\n\*\*|\n```|\Z)', block, re.DOTALL)
        if not instr_match:
            stats['dropped_instr'] += 1
            continue
        raw_instruction = instr_match.group(1).strip().replace('\n', ' ')
        codes = extract_aro_blocks(block)
        if not codes:
            stats['dropped_instr'] += 1
            continue

        instruction = clean_instruction(raw_instruction)
        if not instruction or len(instruction) < 15:
            stats['dropped_instr'] += 1
            continue

        code = codes[0]

        # Reject trivially short outputs
        if count_aro_statements(code) < 2:
            stats['dropped_trivial'] += 1
            print(f'    ~ {chapter_name}: trivial ({count_aro_statements(code)} stmt)', flush=True)
            continue

        check_ok, check_err = aro_check(code)

        # Auto-wrap bare snippets
        if check_ok is False:
            wrap_result, was_wrapped = auto_wrap_aro(code)
            if wrap_result is not None and was_wrapped:
                wrap_ok, wrap_err = aro_check(wrap_result)
                if wrap_ok is True:
                    code = wrap_result
                    check_ok = True
                    wrapped += 1
                    stats['wrapped'] += 1

        # Repair loop
        if check_ok is False:
            fixed_code, check_ok, attempts = repair_aro(
                code, check_err, messages, max_retries=2
            )
            if check_ok is True:
                code = fixed_code
                repaired += 1
                stats['repaired'] += 1
                print(f'    ✓ {chapter_name}: repaired in {attempts} attempt(s)', flush=True)
            else:
                stats['dropped_check'] += 1
                continue

        if save_pair(f'aro_by_example:{chapter_name}', instruction,
                     f'```aro\n{code}\n```',
                     score=1.0 if check_ok is True else 0.8,
                     metadata={'chapter': chapter_name}):
            stats['valid'] += 1
            count += 1
            print(f'  [{count}] {chapter_name}: {instruction[:80]}', flush=True)

print(f'\nPhase 3 done: {count} new pairs from AROByExample ({wrapped} wrapped, {repaired} repaired)')
print(f'  attempted={stats["attempted"]}, valid={stats["valid"]}, wrapped={stats["wrapped"]}, '
      f'repaired={stats["repaired"]}, dropped_check={stats["dropped_check"]}, '
      f'dropped_instr={stats["dropped_instr"]}, dropped_trivial={stats["dropped_trivial"]}')

## Phase 4 — Proposals → Contextualized Training Pairs

The language proposals contain hundreds of ARO code blocks demonstrating specific features.
Many are **bare snippets** without a feature set wrapper (e.g. just `Extract the <user> from the <request: body>.`).

This phase:
- Tries each code block raw against `aro check`
- If it fails, **auto-wraps** bare statements in `(Application-Start: Example) { ... }` and retries
- Filters out non-code content (templates, negative examples, meta-syntax diagrams)
- Generates a natural language instruction for each valid block
- Expected yield improvement: from ~19% to ~58% of proposal code blocks

**Both forms are valid training targets** (issue #380): bare REPL statements are
what `aro repl` and `echo '…' | aro` consume; complete feature sets are what
applications need. Auto-wrapping exists so bare proposal snippets can be
*validated* with `aro check` — the `auto_wrapped` metadata flag records which
pairs are synthetic wrappers, and NB16 caps their share of `code_generation`
pairs at `AUTO_WRAP_MAX_SHARE` (config.py).

In [ ]:
print(f'\n--- Phase 4: Proposals → Contextualized Pairs ---')
count = 0
stats = pipeline_stats['Phase 4']

for prop in kb['proposals']:
    prop_name = prop['name']

    # ── Build a code→(heading, body) lookup from qa_seeds for rich context ──
    seed_context = {}
    for seed in prop.get('qa_seeds', []):
        for code in seed.get('code_examples', []):
            if code.strip():
                seed_context[code.strip()[:60]] = (seed['heading'], seed['body'][:500])

    # ── Use ALL code blocks from the proposal ────────────────────────────────
    all_code_blocks = []
    seen_code = set()
    for code in prop.get('aro_code_blocks', []):
        norm = code.strip()
        if norm and norm not in seen_code:
            seen_code.add(norm)
            all_code_blocks.append(norm)
    for seed in prop.get('qa_seeds', []):
        for code in seed.get('code_examples', []):
            norm = code.strip()
            if norm and norm not in seen_code:
                seen_code.add(norm)
                all_code_blocks.append(norm)

    if not all_code_blocks:
        continue

    for i, code in enumerate(all_code_blocks[:15]):
        if len(code.strip()) < 40:
            continue

        code_key = f'proposal:{prop_name}:{i}'
        stats['attempted'] += 1

        # Try raw code first, then auto-wrap bare snippets
        check_ok, check_err = aro_check(code)
        was_wrapped = False
        output_code = code

        if check_ok is False:
            wrapped, was_wrapped = auto_wrap_aro(code)
            if wrapped is None:
                # Non-code content (template, negative example, meta-syntax)
                stats['dropped_trivial'] += 1
                continue
            check_ok, check_err = aro_check(wrapped)
            if check_ok is True:
                output_code = wrapped
                stats['wrapped'] += 1
            else:
                stats['dropped_check'] += 1
                continue

        if check_ok is not True:
            stats['dropped_check'] += 1
            continue

        # Rich context from matching seed heading
        ctx_key = code.strip()[:60]
        if ctx_key in seed_context:
            heading, body = seed_context[ctx_key]
            context = f'Section "{heading}": {body}'
        elif prop.get('qa_seeds'):
            seed = prop['qa_seeds'][0]
            context = f'Section "{seed["heading"]}": {seed["body"][:300]}'
        else:
            context = ''

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': (
                f'This ARO code example is from specification {prop_name}.\n\n'
                f'Context: {context}\n\n'
                f'```aro\n{output_code}\n```\n\n'
                f'Write ONE natural language instruction (1-2 sentences) that would '
                f'prompt a developer to write exactly this code. '
                f'Do NOT use ARO jargon like "feature set" or angle-bracket syntax. '
                f'Use plain English.\n'
                f'Reply with ONLY the instruction text.'
            )},
        ]

        raw_instruction = chat(messages, max_tokens=120).strip()
        raw_instruction = re.sub(r'^(#+\s*|Instruction:\s*|\*\*[^*]+\*\*:\s*)', '', raw_instruction).strip()
        raw_instruction = raw_instruction.split('\n')[0].strip()

        instruction = clean_instruction(raw_instruction)
        if not instruction or len(instruction) < 15:
            stats['dropped_instr'] += 1
            continue

        if save_pair(code_key, instruction, f'```aro\n{output_code}\n```',
                     score=1.0,
                     metadata={'proposal': prop_name, 'block_index': i,
                               'auto_wrapped': was_wrapped}):
            stats['valid'] += 1
            count += 1
            tag = ' [wrapped]' if was_wrapped else ''
            if count % 20 == 0:
                print(f'  [{count}]{tag} {prop_name}[{i}]: {instruction[:70]}', flush=True)

print(f'\nPhase 4 done: {count} new pairs from proposals ({stats["wrapped"]} auto-wrapped)')
print(f'  attempted={stats["attempted"]}, valid={stats["valid"]}, wrapped={stats["wrapped"]}, '
      f'dropped_check={stats["dropped_check"]}, dropped_instr={stats["dropped_instr"]}, '
      f'dropped_trivial={stats["dropped_trivial"]}')

## Phase 5 — Instruction Variants

For each high-quality pair (score ≥ 1.0), generate 2 alternative instruction phrasings.
This multiplies the dataset without new code generation — same validated code,
diverse instruction styles. Teaches the model to respond to many ways of asking
the same thing.

In [ ]:
print(f'\n--- Phase 5: Instruction Variants ---')
variant_count = 0
stats = pipeline_stats['Variants']

# Collect all NB04 pairs saved so far (read from JSONL)
base_pairs = []
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    p = json.loads(line)
                    if p.get('notebook') == 'NB04' and p.get('score', 0) >= 1.0:
                        base_pairs.append(p)
                except Exception:
                    pass

# Only generate variants for pairs with substantial code (not trivial one-liners)
substantial_pairs = [p for p in base_pairs
                     if len(p.get('output', '')) > 200
                     and 'variant' not in p.get('source', '')]

print(f'  {len(substantial_pairs)} substantial base pairs eligible for variants')

for p in substantial_pairs:
    original_instr = p['instruction']
    source = p['source']
    stats['attempted'] += 2  # we ask for 2 variants per pair

    messages = [
        {'role': 'system', 'content': (
            'You are helping create diverse training data. '
            'Given an instruction, rephrase it in 2 different ways. '
            'Keep the same meaning but vary the style:\n'
            '- Use different openings (Build..., I need..., Set up..., How do I..., '
            'Write code to..., Can you make...)\n'
            '- Some formal, some casual\n'
            '- Do NOT use ARO jargon like "feature set", "business activity", '
            'or angle-bracket syntax\n'
            '- Keep each variant 1-3 sentences'
        )},
        {'role': 'user', 'content': (
            f'Original instruction:\n"{original_instr}"\n\n'
            f'Write exactly 2 rephrased variants. Format:\n'
            f'VARIANT 1: <rephrased instruction>\n'
            f'VARIANT 2: <rephrased instruction>'
        )},
    ]

    output_text = chat(messages, max_tokens=300)

    variants = re.findall(r'VARIANT\s*\d+:\s*(.+?)(?=\nVARIANT|\Z)', output_text, re.DOTALL)
    if not variants:
        # Try fallback parsing
        lines = [l.strip() for l in output_text.strip().split('\n')
                 if l.strip() and len(l.strip()) > 20
                 and not l.strip().startswith('Original')]
        variants = lines[:2]

    for vi, variant in enumerate(variants[:2]):
        variant = variant.strip().strip('"').strip()
        cleaned = clean_instruction(variant)
        if not cleaned or len(cleaned) < 20:
            stats['dropped_instr'] += 1
            continue
        if cleaned.lower() == original_instr.lower():
            stats['dropped_instr'] += 1
            continue

        # Re-run the semantic-alignment gate on the (variant instruction,
        # original code) pair before saving (issue #381). Variants inherit
        # score=1.0, so a misaligned rewrite would teach a spurious
        # instruction→code mapping with full confidence.
        aligned, reason = semantic_alignment_check(cleaned, p['output'], _nb03_judge_chat)
        if not aligned:
            stats['dropped_misaligned'] += 1
            if stats['dropped_misaligned'] <= 5:
                print(f'    x variant misaligned [{source}]: {reason[:100]}', flush=True)
            continue

        variant_source = f'{source}:variant{vi+1}'
        if save_pair(variant_source, cleaned, p['output'], score=1.0,
                     metadata={**p.get('metadata', {}), 'variant_of': source},
                     skip_gate=True):
            stats['valid'] += 1
            variant_count += 1

    if variant_count % 40 == 0 and variant_count > 0:
        print(f'  [{variant_count}] variants generated...', flush=True)

print(f'\nPhase 5 done: {variant_count} instruction variants')
print(f'  attempted={stats["attempted"]}, valid={stats["valid"]}, '
      f'dropped_instr={stats["dropped_instr"]}, '
      f'dropped_misaligned={stats["dropped_misaligned"]}')

## Summary

All pairs written to `knowledge_pairs.jsonl`.

**Next steps:**
- Run **`05_warmstart_finetune.ipynb`** — warm-start fine-tune on these pairs
- Run **`08_book_qa_pairs.ipynb`** — generate more Q&A pairs from all Book chapters

In [ ]:
all_pairs = []
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    p = json.loads(line)
                    # Only count NB04 pairs for this notebook's summary
                    if p.get('notebook') == 'NB04':
                        all_pairs.append(p)
                except Exception:
                    pass

sources = Counter(p.get('source', 'unknown').split(':')[0] for p in all_pairs)
scores  = Counter(round(p.get('score', 1.0), 1) for p in all_pairs)

print(f'\nTotal NB04 knowledge pairs: {len(all_pairs)}')
print('\nBy source:')
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:30s}: {n}')
print('\nBy score:')
for score, n in sorted(scores.items(), key=lambda x: -x[0]):
    print(f'  {score}: {n}')

# ── Auto-wrap tracking (issue #380) ─────────────────────────────────────────
# Report auto-wrapped vs. real pairs per source so nothing about the
# synthetic-wrapper share stays invisible. NB16 enforces the cap.
wrapped_by_src, real_by_src = Counter(), Counter()
for p in all_pairs:
    prefix = p.get('source', 'unknown').split(':')[0]
    if p.get('metadata', {}).get('auto_wrapped'):
        wrapped_by_src[prefix] += 1
    else:
        real_by_src[prefix] += 1

print('\nAuto-wrapped vs real pairs (per source):')
for src in sorted(set(wrapped_by_src) | set(real_by_src)):
    w, r = wrapped_by_src[src], real_by_src[src]
    share = w / max(1, w + r)
    flag = '  ⚠ above NB16 cap' if share > AUTO_WRAP_MAX_SHARE else ''
    print(f'  {src:<20}: {w:4d} wrapped / {r:4d} real  ({share:.0%} wrapped){flag}')
total_wrapped = sum(wrapped_by_src.values())
print(f'\nTotal auto-wrapped: {total_wrapped}/{len(all_pairs)} '
      f'({total_wrapped / max(1, len(all_pairs)):.0%}) — NB16 caps auto-wrapped '
      f'code_generation pairs at {AUTO_WRAP_MAX_SHARE:.0%} (AUTO_WRAP_MAX_SHARE).')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from collections import Counter
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '04_llm_knowledge_extraction.png'

# ── Figure: Pipeline funnel (attempted → valid → repaired/wrapped → dropped) ─
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [3, 2]})

phases = ['Phase 1', 'Phase 2', 'Phase 3', 'Phase 4', 'Variants']
phase_labels = [
    'Phase 1\nReal examples',
    'Phase 2\nLanguage Guide',
    'Phase 3\nAROByExample',
    'Phase 4\nProposals',
    'Phase 5\nVariants',
]

valid_first  = []
repaired_cnt = []
wrapped_cnt  = []
drop_check   = []
drop_instr   = []
drop_trivial = []

for phase in phases:
    s = pipeline_stats[phase]
    # valid includes repaired + wrapped (both are subsets of valid)
    vf = s['valid'] - s['repaired'] - s['wrapped']
    valid_first.append(max(vf, 0))
    repaired_cnt.append(s['repaired'])
    wrapped_cnt.append(s['wrapped'])
    drop_check.append(s['dropped_check'])
    drop_instr.append(s['dropped_instr'])
    drop_trivial.append(s['dropped_trivial'])

x = np.arange(len(phases))
width = 0.6

ax = axes[0]
# Stack: valid 1st try → wrapped → repaired → drop:trivial → drop:check → drop:instr
cum = [0] * len(phases)
b1 = ax.bar(x, valid_first,  width, bottom=cum, label='Valid (1st try)', color='#2ecc71')
cum = [a+b for a,b in zip(cum, valid_first)]
b2 = ax.bar(x, wrapped_cnt,  width, bottom=cum, label='Auto-wrapped', color='#1abc9c')
cum = [a+b for a,b in zip(cum, wrapped_cnt)]
b3 = ax.bar(x, repaired_cnt, width, bottom=cum, label='Repaired', color='#3498db')
cum = [a+b for a,b in zip(cum, repaired_cnt)]
b4 = ax.bar(x, drop_trivial, width, bottom=cum,
            label='Dropped: trivial/non-code', color='#f39c12', alpha=0.7)
cum = [a+b for a,b in zip(cum, drop_trivial)]
b5 = ax.bar(x, drop_check, width, bottom=cum,
            label='Dropped: aro check', color='#e74c3c', alpha=0.7)
cum = [a+b for a,b in zip(cum, drop_check)]
b6 = ax.bar(x, drop_instr, width, bottom=cum,
            label='Dropped: bad instruction', color='#95a5a6', alpha=0.7)

# Label each bar with valid/attempted
for i in range(len(phases)):
    total = (valid_first[i] + wrapped_cnt[i] + repaired_cnt[i]
             + drop_check[i] + drop_instr[i] + drop_trivial[i])
    valid_total = valid_first[i] + wrapped_cnt[i] + repaired_cnt[i]
    if total > 0:
        ax.text(i, total + 0.5, f'{valid_total}/{total}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(phase_labels, fontsize=8)
ax.set_ylabel('Pairs')
total_valid = sum(valid_first) + sum(wrapped_cnt) + sum(repaired_cnt)
total_attempted = total_valid + sum(drop_check) + sum(drop_instr) + sum(drop_trivial)
ax.set_title(f'Pipeline Funnel — {total_valid} valid of {total_attempted} attempted',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=7.5)
ax.grid(axis='y', alpha=0.3)

# Right panel: summary donut chart
ax2 = axes[1]
sizes = [sum(valid_first), sum(wrapped_cnt), sum(repaired_cnt),
         sum(drop_trivial), sum(drop_check), sum(drop_instr)]
labels = ['Valid (1st try)', 'Auto-wrapped', 'Repaired',
          'Drop: non-code', 'Drop: aro check', 'Drop: instruction']
colors = ['#2ecc71', '#1abc9c', '#3498db', '#f39c12', '#e74c3c', '#95a5a6']
nonzero = [(s, l, c) for s, l, c in zip(sizes, labels, colors) if s > 0]
if nonzero:
    sizes_nz, labels_nz, colors_nz = zip(*nonzero)
    wedges, texts, autotexts = ax2.pie(
        sizes_nz, labels=labels_nz, colors=colors_nz,
        autopct=lambda pct: f'{int(round(pct * sum(sizes_nz) / 100))}',
        startangle=90, pctdistance=0.75,
        wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2),
        textprops={'fontsize': 9},
    )
    for at in autotexts:
        at.set_fontweight('bold')
        at.set_fontsize(10)
ax2.set_title(f'Overall: {total_valid} pairs ({100*total_valid/max(total_attempted,1):.0f}% yield)',
              fontsize=12, fontweight='bold')

fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

# Print stats table
print(f'\n{"Phase":<18} {"Attempted":>9} {"Valid":>6} {"Wrapped":>8} {"Repaired":>8} '
      f'{"Drop✗":>6} {"Drop~":>6} {"DropI":>6}')
print('-' * 78)
for phase in phases:
    s = pipeline_stats[phase]
    print(f'{phase:<18} {s["attempted"]:>9} {s["valid"]:>6} {s["wrapped"]:>8} {s["repaired"]:>8} '
          f'{s["dropped_check"]:>6} {s["dropped_trivial"]:>6} {s["dropped_instr"]:>6}')
totals = {k: sum(pipeline_stats[p][k] for p in phases)
          for k in ['attempted', 'valid', 'wrapped', 'repaired', 'dropped_check', 'dropped_trivial', 'dropped_instr']}
print('-' * 78)
print(f'{"TOTAL":<18} {totals["attempted"]:>9} {totals["valid"]:>6} {totals["wrapped"]:>8} {totals["repaired"]:>8} '
      f'{totals["dropped_check"]:>6} {totals["dropped_trivial"]:>6} {totals["dropped_instr"]:>6}')

In [ ]:
# ── CSV export for NB04 pairs only ───────────────────────────────────────────
import csv

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
csv_path = _run_dir / '04_llm_knowledge_extraction.csv'

# Filter to NB04 pairs only
nb03_pairs = [p for p in all_pairs if p.get('notebook') == 'NB04']

def rescore_pair(pair):
    """Assign a real quality score based on content analysis."""
    output = pair.get('output', '')
    has_code = bool(extract_aro_blocks(output))
    score = pair.get('score', 1.0)

    if has_code and score >= 1.0:
        return 1.0
    elif has_code:
        return 0.7
    else:
        instr = pair.get('instruction', '').lower()
        _aro_terms = {'application', 'aro', 'action', 'extract', 'return',
                      'compute', 'retrieve', 'store', 'log', 'emit', 'keepalive'}
        if len(output) > 100 and any(t in instr for t in _aro_terms):
            return 0.8
        return 0.7

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['phase', 'instruction_preview', 'quality_score', 'has_code', 'is_variant', 'source'])
    for p in nb03_pairs:
        raw_src = p.get('source', 'unknown').split(':')[0]
        is_variant = 'variant' in p.get('source', '')
        phase = {
            'example': 'Phase 1 Real examples',
            'book': 'Phase 2 Language Guide',
            'aro_by_example': 'Phase 3 AROByExample',
            'proposal': 'Phase 4 Proposals',
        }.get(raw_src, raw_src)
        if is_variant:
            phase += ' (variant)'
        score = rescore_pair(p)
        has_code = bool(extract_aro_blocks(p.get('output', '')))
        writer.writerow([
            phase,
            p.get('instruction', '')[:150],
            score,
            has_code,
            is_variant,
            p.get('source', ''),
        ])

print(f'CSV saved: {csv_path} ({len(nb03_pairs)} NB04 rows)')

# Score distribution
real_scores = Counter(rescore_pair(p) for p in nb03_pairs)
print('\nNB03 score distribution:')
for score, n in sorted(real_scores.items(), key=lambda x: -x[0]):
    print(f'  {score}: {n}')

# Variant stats
originals = sum(1 for p in nb03_pairs if 'variant' not in p.get('source', ''))
variants = sum(1 for p in nb03_pairs if 'variant' in p.get('source', ''))
print(f'\nOriginals: {originals}, Variants: {variants}, Total: {len(nb03_pairs)}')